# MoE: маршрутизация между экспертами

Top-k routing, дисбаланс и отличие общего размера от активного.

> Все примеры этого урока работают на CPU, не скачивают модели и используют искусственные данные. Они объясняют механизм; качество настоящего обучения проверяется отдельно.

## Что важно понять

Mixture of Experts выбирает часть экспертов для каждого токена. Общий размер включает всех экспертов, а вычисления зависят от числа активных. Реальная экономия зависит от реализации dispatcher, batch, коммуникаций и железа.

Ниже наглядный top-1 router. Дискретный argmax не даёт градиента через выбор эксперта: production MoE использует подходящую схему взвешивания и балансировки. Пример нужен для изучения маршрутов и не выдаёт себя за полноценный MoE trainer.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Запустите эксперимент

Запускайте ячейки по порядку. Код можно менять; Shift + Enter выполняет текущую ячейку.

In [ ]:
tokens = torch.randn(32, 8)
router = nn.Linear(8, 4)
experts = nn.ModuleList([nn.Sequential(nn.Linear(8, 16), nn.GELU(), nn.Linear(16, 8)) for _ in range(4)])
probabilities = router(tokens).softmax(-1)
routes = probabilities.argmax(-1)
output = torch.zeros_like(tokens)
for index, expert in enumerate(experts):
    selected = routes == index
    if selected.any():
        output[selected] = expert(tokens[selected])
counts = torch.bincount(routes, minlength=4)
print('tokens per expert:', counts.tolist())
print('all expert parameters:', sum(p.numel() for p in experts.parameters()))
print('output shape:', output.shape)
plt.bar(range(4), counts.tolist(), color='#d73535'); plt.xlabel('expert'); plt.show()

## Попробуйте сами

Сместите bias router к одному эксперту и посмотрите на распределение. Почему почти неиспользуемые эксперты ухудшают обучение? Какие величины нужно логировать кроме общего loss?

Сохраните копию ноутбука и запишите, что изменили, чего ожидали и что измерили.

## Продолжить в студии

Кнопка «Открыть раздел студии» над уроком ведёт к связанным инструментам. Учебный код сам не запускает большие тренировки. Перед реальным запуском проверьте данные, ресурсы и поддержку метода у выбранной модели.

## Источники

- [Switch Transformers](https://arxiv.org/abs/2101.03961)